In [15]:
!pip -q install monai zarr numcodecs

In [16]:
# ============================================================
# Imports
# ============================================================

import os
import gc
import random
import warnings
from pathlib import Path

import cv2
import zarr
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision.models import (
    efficientnet_b0,
    EfficientNet_B0_Weights
)

from torch.amp import autocast
from torch.amp import GradScaler

warnings.filterwarnings("ignore")

In [17]:
# ============================================================
# Configuration
# ============================================================

SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

AMP = DEVICE == "cuda"

DATASET_PATH = Path(
    "/kaggle/input/competitions/biohub-cell-tracking-during-development"
)

TRAIN_PATH = DATASET_PATH / "train"

TEST_PATH = DATASET_PATH / "test"

OUTPUT_PATH = Path("./output")

OUTPUT_PATH.mkdir(exist_ok=True)

print("Device :", DEVICE)

Device : cpu


In [18]:
# ============================================================
# Hyperparameters
# ============================================================

IMAGE_SIZE = 256

NUM_SLICES = 5

HALF = NUM_SLICES // 2

CROP_SIZE = 256

SIGMA = 3

BATCH_SIZE = 32

EPOCHS = 15

PATIENCE = 4

LR = 1e-3

WEIGHT_DECAY = 1e-4

NUM_WORKERS = 2

MIN_DELTA = 1e-3

In [19]:
# ============================================================
# Seed
# ============================================================

def seed_everything(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.benchmark = True

    torch.backends.cudnn.deterministic = False


seed_everything(SEED)

In [20]:
# ============================================================
# Discover Dataset
# ============================================================

train_zarr = sorted(TRAIN_PATH.glob("*.zarr"))

train_geff = sorted(TRAIN_PATH.glob("*.geff"))

print(f"Volumes     : {len(train_zarr)}")

print(f"Annotations : {len(train_geff)}")

assert len(train_zarr) == len(train_geff)

for z,g in zip(train_zarr[:5], train_geff[:5]):

    print()

    print(z.name)

    print(g.name)# ============================================================
# Pair Files
# ============================================================

samples = []

for zarr_file in train_zarr:

    geff_file = TRAIN_PATH / f"{zarr_file.stem}.geff"

    if geff_file.exists():

        samples.append({

            "name": zarr_file.stem,

            "zarr": zarr_file,

            "geff": geff_file

        })

print("Training Samples :", len(samples))

Volumes     : 199
Annotations : 199

44b6_0113de3b.zarr
44b6_0113de3b.geff

44b6_0b24845f.zarr
44b6_0b24845f.geff

44b6_0c582fdc.zarr
44b6_0c582fdc.geff

44b6_0db75fae.zarr
44b6_0db75fae.geff

44b6_12dfb391.zarr
44b6_12dfb391.geff
Training Samples : 199


In [21]:
# ============================================================
# Gaussian Kernel
# ============================================================

KERNEL_SIZE = 21

radius = KERNEL_SIZE // 2

x = np.arange(KERNEL_SIZE) - radius

xx, yy = np.meshgrid(x, x)

GAUSSIAN_KERNEL = np.exp(

    -(xx**2 + yy**2)

    /

    (2 * SIGMA**2)

).astype(np.float32)

GAUSSIAN_KERNEL /= GAUSSIAN_KERNEL.max()

print(GAUSSIAN_KERNEL.shape)

(21, 21)


In [22]:
# ============================================================
# AMP
# ============================================================

scaler = GradScaler(

    "cuda",

    enabled=AMP

)

In [23]:
# ============================================================
# Cleanup
# ============================================================

gc.collect()

if DEVICE == "cuda":

    torch.cuda.empty_cache()

print("Notebook Ready.")

Notebook Ready.


In [24]:
# ============================================================
# Read all annotations
# ============================================================

metadata = []

for sample in tqdm(samples):

    gt = zarr.open(sample["geff"], mode="r")

    t = np.asarray(gt["nodes"]["props"]["t"]["values"])
    z = np.asarray(gt["nodes"]["props"]["z"]["values"])
    y = np.asarray(gt["nodes"]["props"]["y"]["values"])
    x = np.asarray(gt["nodes"]["props"]["x"]["values"])

    df = pd.DataFrame({
        "t": t,
        "z": z,
        "y": y,
        "x": x
    })

    metadata.append({

        "name": sample["name"],

        "zarr": sample["zarr"],

        "geff": sample["geff"],

        "nodes": df

    })

print(f"Loaded {len(metadata)} samples")

  0%|          | 0/199 [00:00<?, ?it/s]

Loaded 199 samples


In [25]:
# ============================================================
# Compute normalization statistics
# ============================================================

movie_stats = {}

for sample in tqdm(metadata):

    volume = zarr.open(sample["zarr"], mode="r")["0"]

    depth = volume.shape[0]

    idx = np.linspace(
        0,
        depth - 1,
        min(depth, 8),
        dtype=int
    )

    pixels = np.concatenate([
        volume[i].ravel()
        for i in idx
    ])

    movie_stats[sample["name"]] = {

        "p1": np.percentile(pixels, 1),

        "p99": np.percentile(pixels, 99.8)

    }

print("Normalization statistics computed.")

  0%|          | 0/199 [00:00<?, ?it/s]

Normalization statistics computed.


In [26]:
# ============================================================
# Build training records
# ============================================================

records = []

for sample in metadata:

    nodes = sample["nodes"]

    grouped = nodes.groupby(["t", "z"])

    for (t, z), group in grouped:

        records.append({

            "sample": sample["name"],

            "zarr": sample["zarr"],

            "time": int(t),

            "z": int(z),

            "nodes": group.reset_index(drop=True)

        })

print(f"Training records: {len(records)}")

Training records: 122195


In [27]:
# ============================================================
# Train / Validation Split
# ============================================================

sample_names = np.array([r["sample"] for r in records])

unique_samples = np.unique(sample_names)

rng = np.random.default_rng(SEED)

rng.shuffle(unique_samples)

split = int(0.8 * len(unique_samples))

train_samples = set(unique_samples[:split])
val_samples = set(unique_samples[split:])

train_records = [
    r for r in records
    if r["sample"] in train_samples
]

val_records = [
    r for r in records
    if r["sample"] in val_samples
]

print(f"Train records : {len(train_records)}")
print(f"Validation records : {len(val_records)}")

Train records : 99226
Validation records : 22969


In [28]:
# ============================================================
# Sanity check
# ============================================================

print(train_records[0].keys())

print(train_records[0]["nodes"].head())

print(movie_stats[train_records[0]["sample"]])

dict_keys(['sample', 'zarr', 'time', 'z', 'nodes'])
   t   z    y    x
0  0  63  222  249
{'p1': np.float64(30.0), 'p99': np.float64(1813.0)}


In [29]:
sample = metadata[0]

vol = zarr.open(sample["zarr"], mode="r")

print(type(vol))
print(vol.tree())

<class 'zarr.core.group.Group'>
/
└── 0 (100, 64, 256, 256) uint16

